Sarcastic Webpage Summarizer

This notebook uses a locally-running LLM (via Ollama + phi3) to fetch and summarize webpage content — with a sarcastic twist in the tone.

As an example, I'm summarizing the model card for hasoc-student-distilbert, a distilled hate speech detection model I built as part of my HASOC 2021 code-mixed (Hindi-English) hate speech classification project. It's a DistilBERT-based "student" model trained to be a smaller, faster alternative to a larger XLM-R "teacher" model — achieving ~96% of the teacher's performance at roughly 4x faster inference and about half the parameter count.

🔗 Try the live demo here: https://huggingface.co/spaces/Bharath2kk5/code-mixed-hate-speech-demo?logs=container

In [ ]:
import requests
from bs4 import BeautifulSoup
from openai import OpenAI

ollama_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

# Step 1: Fetch and extract webpage text
url = "https://huggingface.co/Bharath2kk5/hasoc-student-distilbert"

headers = {"User-Agent": "Mozilla/5.0"}
response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.content, "html.parser")

# Remove irrelevant tags
for tag in soup(["script", "style", "img", "input"]):
    tag.decompose()

page_text = soup.get_text(separator="\n", strip=True)

# Step 2: Build prompts
system_prompt = "You are a helpful assistant that summarizes webpages concisely using a sarcastic tone, capturing key action items and dates."

user_prompt = f"""
Summarize this webpage content:

{page_text}
"""

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
]

# Step 3: Call the model
completion = ollama_client.chat.completions.create(model="phi3", messages=messages)

# Step 4: Print the result
print(completion.choices[0].message.content)